# <span style="color:Thistle">Operaciones sobre datos en Pandas</span>

#### <span style="color:beige">Sesión 3</span>
---


Uno de los elementos esenciales de NumPy es la capacidad de realizar operaciones rápidas elemento a elemento como por ejemplo suma, resta, multiplicación, etc.. <span style="color:Gold">Pandas hereda gran parte de esta funcionalidad de NumPy, y las **ufuncs**\* (funciones universales de NumPy) son clave para ello.</span>

Esto significa que mantener el contexto de los datos y combinar datos de distintas fuentes (tareas potencialmente propensas a errores con arrays NumPy puros) se convierten en tareas prácticamente infalibles con Pandas. 

Además, veremos que existen operaciones bien definidas entre estructuras `Series` unidimensionales y estructuras `DataFrame` bidimensionales.

*Ufuncs: Las ufuncs (universal functions o funciones universales) son funciones de NumPy que operan elemento por elemento sobre arrays, permitiendo realizar operaciones vectorizadas de manera muy eficiente.

In [1]:
import pandas as pd
import numpy as np

## Ufuncs: Alineación de índices

En las operaciones binarias sobre dos objetos `Series` o `DataFrame`, <span style="color:Gold">Pandas alineará los índices durante la operación.</span> Esto resulta muy conveniente cuando se trabaja con datos incompletos, como veremos en los ejemplos a continuación.

### Alineación de índices en Series

Como ejemplo, supongamos que combinamos dos fuentes de datos distintas y solo tenemos los tres estados de EE. UU. con mayor *superficie* y los tres con mayor *población*:

In [2]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
poblacion = pd.Series({'California': 38332521, 'Texas': 26448193,
                        'New York': 19651127}, name='poblacion')

area, poblacion

(Alaska        1723337
 Texas          695662
 California     423967
 Name: area, dtype: int64,
 California    38332521
 Texas         26448193
 New York      19651127
 Name: poblacion, dtype: int64)

Veamos qué ocurre al dividirlos para calcular la densidad de población:

In [3]:
densidad = poblacion / area
densidad

Alaska              NaN
California    90.413926
New York            NaN
Texas         38.018740
dtype: float64

<span style="color:Gold">Cualquier elemento para el cual uno u otro no tiene valor se marca con `NaN` ("Not a Number"), que es la forma en que Pandas representa los datos ausentes.</span>

Al sumar cualquier número con `NaN`, el resultado es `NaN`.

Vamos a sumar dos Series para comprobar que los valores ausentes se rellenan con NaN de forma predeterminada:

In [4]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])
# 0 --> 2 + NaN = Nan
# 1 --> 4 + 1 = 5
# 2 --> 6 +3 = 9
# 3 --> NaN + 5 = NaN
A + B

0    NaN
1    5.0
2    9.0
3    NaN
dtype: float64

Podemos cambiar este comportamiento, y hacer que sume un número cuando esté falte.

Por ejemplo, llamar a `A.add(B)` es equivalente a `A + B`.

Y la ventaja es que podemos especificar el valor de relleno para los elementos ausentes (que considera NaN) en `A` o `B` con esta llamada:

In [6]:
A.add(B, fill_value=0)

0    2.0
1    5.0
2    9.0
3    5.0
dtype: float64

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Qué pasa si hacemos B.add(A, fill_value=0)?
</div>


<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Qué cambia si hacemos la llamada con fill_value=1?
</div>

### Alineación de índices en DataFrame

Un tipo similar de alineación ocurre tanto en columnas como en índices al realizar operaciones sobre `DataFrames`:

In [7]:
rng = np.random.RandomState(42)

matrizA = pd.DataFrame(rng.randint(0, 20, (2, 2)), # Creamos una matrix de 2x2 con valores aleatorios entre 0 y 20
                 columns=list('AB'))
matrizA

,A,B
0,6,19
1,14,10


In [8]:
matrizB = pd.DataFrame(rng.randint(0, 10, (3, 3)), # Creamos una matriz de 3x3 con valores aleatorios entre 0 y 10
                 columns=list('BAC'))
matrizB

,B,A,C
0,7,4,6
1,9,2,6
2,7,4,3


In [9]:
matrizA + matrizB

,A,B,C
0,10.0,26.0,NaN
1,16.0,19.0,NaN
2,NaN,NaN,NaN


<div class="alert alert-block alert-success">
Fijate que en la 1º matriz las columnas son AB y en la 2º matriz el orden es BAC. Pero al sumarlas, las muestra como ABC
</div>


<span style="color:Gold">Los índices se alinean correctamente SIN DEPENDER de su orden en los dos objetos, y los índices del resultado están ordenados.</span>

Al igual que con `Series`, podemos usar el método del objeto y pasar un `fill_value` para reemplazar las entradas ausentes. Aquí rellenaremos con la media de todos los valores en `A` (calculada apilando primero las filas de `A`):

In [10]:
fill = matrizA.stack().mean()
matrizA.add(matrizB, fill_value=fill)

,A,B,C
0,10.00,26.00,18.25
1,16.00,19.00,18.25
2,16.25,19.25,15.25


<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Qué hace matrizA.stack()? 

Ejecutalo y explicalo con tus palabras.
</div>

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Y matrizA.stack().mean()?
</div>

La siguiente tabla lista los operadores de Python y sus métodos equivalentes en Pandas:

| Operador Python | Método(s) en Pandas |
|-----------------|---------------------|
| `+`             | `add()` |
| `-`             | `sub()`, `subtract()` |
| `*`             | `mul()`, `multiply()` |
| `/`             | `truediv()`, `div()`, `divide()` |
| `//`            | `floordiv()` |
| `%`             | `mod()` |
| `**`            | `pow()` |

## Ufuncs: Operaciones entre DataFrame y Series

Al realizar operaciones entre un `DataFrame` y un `Series`, la alineación de índices y columnas se mantiene de forma similar. Las operaciones entre un `DataFrame` y un `Series` son análogas a las operaciones entre un array NumPy bidimensional y uno unidimensional.

Consideremos una operación común: encontrar la diferencia entre un array bidimensional y una de sus filas:

In [11]:
npA = rng.randint(10, size=(3, 4))
npA

array([[7, 7, 2, 5],
       [4, 1, 7, 5],
       [1, 4, 0, 9]], dtype=int32)

In [12]:
npA - npA[0]

array([[ 0,  0,  0,  0],
       [-3, -6,  5,  0],
       [-6, -3, -2,  4]], dtype=int32)

Según las reglas de *broadcasting* de NumPy, la resta entre un array bidimensional y una de sus filas se aplica fila a fila.

En Pandas, se hace de forma similar por defecto, fila a fila:

In [14]:
df = pd.DataFrame(npA, columns=list('QRST'))
df - df.iloc[0]

,Q,R,S,T
0,0,0,0,0
1,-3,-6,5,0
2,-6,-3,-2,4


<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Qué hace iloc?
</div>